# Access NISAR GUNW and Copernicus GLO-30 DEM

<br>

This notebook demonstrates how to search and download a stack of [NISAR Geocoded Unwrapped Interferograms (GUNWs)](https://nisar-docs.asf.alaska.edu/gunw/) and the [Copernicus GLO-30 DEM](https://portal.opentopography.org/raster?opentopoID=OTSDEM.032021.4326.3) in preparation for processing with MintPy. Note that the NISAR GUNW dataset includes temporal nearest-neighbor interferometric pairs only, and therefor does not provide an ideal, highly-redunant SBAS network.

:::{hint} Apply to be a HyP3 NISAR GUNW-like Product Beta-Tester!

The Alaska Satellite Facility (ASF) is developing a [HyP3](https://hyp3-docs.asf.alaska.edu/hyp3-docs/guides/burst_insar_product_guide/) on-demand processing service to generate NISAR Geocoded Unwrapped Interferogram-like (GUNW-like) products from custom pairs of [NISAR RSLCs](https://nisar-docs.asf.alaska.edu/rslc/), and we are seeking beta testers. These custom GUNWs are not standard NISAR products produced by the mission team at JPL -- They are custom products processed by ASF using a workflow and software ([ISCE3](https://github.com/isce-framework/isce3)) that produces comparable products.

**This HyP3 plugin allows for the creation of highly-redundant SBAS networks of GUNW-like interferograms**

[NISAR Custom GUNW HyP3 On-Demand Processing Application Form](https://docs.google.com/forms/d/e/1FAIpQLSckXT2lOGo1qDDa6AFqftfAtirLoZ76xqI5l8sTjsMmikWG7Q/viewform?usp=header)
:::

<hr>

:::{note} **Did you find a bug? Do you have a feature request?**
:::{figure} github_issues.png
:alt: GitHub logo above the word Issues
:width: 10%
:align: left
:target: https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues

Explore GitHub Issues on this Jupyter Book's GitHub repository. Find solutions, add to the discussion, or start a new bug report or feature request: <a href="https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues" target="_blank" rel="noopener noreferrer">opensarlab_MintPy_Recipe_Book Issues</a>
:::


:::{note} **Have a question related to SAR, ASF data access, or performing SBAS time series analyses with MintPy?**
:::{figure} ASF_support_logo.png
:alt: ASF logo
:width: 10%
:align: left
:target: https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues

Contact ASF User Support: <a href="mailto:uso@asf.alaska.edu" target="_blank" rel="noopener noreferrer">uso@asf.alaska.edu</a>
:::

<hr>

## 0. Initialize the user-defined variables for this notebook

Update the variables below to initalize your NISAR GUNW search parameters and create a directory in which to store your data.

Use [Vertex](https://nisar-docs.asf.alaska.edu/vertex/) to preview the data you wish to access and identify the values to use for the search parameters below.

In [ ]:
from datetime import datetime
from pathlib import Path

start_date = datetime(2025, 5, 22)
end_date = datetime(2028, 12, 5)

# POINT or POLYGON as WKT (well-known-text)
area_of_interest = "POLYGON((-90.3933 29.1394,-89.4273 29.1394,-89.4273 30.0661,-90.3933 30.0661,-90.3933 29.1394))" # Tip: Copy/paste WKT from a Vertex search

frame = 74
relativeOrbit = 156

# Create a directory in which to store your data 
data_dir = Path.cwd() / "MintPy_example_GUNW_data"
data_dir.mkdir(exist_ok=True)
print(f'data_dir: {data_dir}')

## 1a. Search for GNISAR GUNW data with asf_search

In [ ]:
import asf_search as asf
from getpass import getpass

session = asf.ASFSession()
session.auth_with_creds(input('EDL Username'), getpass('EDL Password'))

opts=asf.ASFSearchOptions(**{
    "maxResults": 250,
    "intersectsWith": area_of_interest,
    "start": start_date,
    "end": end_date,
    "processingLevel": [
        "GUNW"
    ],
    "dataset": [
        "NISAR"
    ],
    "productionConfiguration": [
        "PR"
    ],
    "dataMaturity" : "PROVISIONAL",
    'session': session,
    "frame": frame,
    "relativeOrbit": relativeOrbit,
})

pattern = r'^(?!.*QA_STATS).*' # regex to remove static data files from results
response = asf.search(opts=opts)
hdf5_urls = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
hdf5_urls

## 1b. Retain only the latest GUNW product releases

NISAR products are occasionaly re-released using the same processor version for a variety of possible reasons. These products will maintain the same CRID and have an incremented CTR. See [NISAR naming convention documentation](https://nisar-docs.asf.alaska.edu/naming-conventions/) for details.

The cell below retains only the most recent release of each NISAR GUNW.

In [ ]:
import re

def keep_latest_gunw(urls):
    latest = {}

    for url in urls:
        dates = re.findall(r"\d{8}T\d{6}", url)
        pair = (dates[0][:8], dates[2][:8])
        version = int(re.search(r"_(\d{3})\.h5", url).group(1))

        if pair not in latest or version > latest[pair][0]:
            latest[pair] = (version, url)

    return [url for version, url in latest.values()]

keep_latest_gunw(hdf5_urls)

## 2. Download the NISAR GUNWs

In [ ]:
asf.download_urls(hdf5_urls, data_dir, session=session)

## 3a. Find a GUNW's bounding box and EPSG

This information will be used for subsetting and reprojecting the COP GLO-30 DEM

In [ ]:
from shapely import wkt
import h5py

gunw_path = list(data_dir.glob("*GUNW*.h5"))[0]

with h5py.File(gunw_path) as f:
    polygon = wkt.loads(f["/science/LSAR/identification/boundingPolygon"][()].decode())
    epsg =  f["/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/projection"][()].item()

bbox = polygon.bounds  # west, south, east, north

print(f"bbox: {bbox}")
print(f"EPSG: {epsg}")

## 3b. (Optional) Inspect one of the GUNWs with Xarray

Xarray provides an interactive HTML UI for exploring HDF5 data structures in a Jupyter Notebook.

Uncomment and run the cell below to view one of the downloaded GUNWs.

In [ ]:
# import xarray as xr

# dt = xr.open_datatree(
#         open(gunw_path, "rb"),
#         engine="h5netcdf",
#         decode_timedelta=False,
#         phony_dims="access",
#         chunks="auto",
# )
# display(dt)

## 4. Download a subset Copernicus GLO-30 DEM with sardem

In [ ]:
import sardem.dem as dem

dem_4326 = data_dir / "DEM.tif"

dem.main(
    output_name=dem_4326,
    bbox=bbox,
    data_source="COP"
)

## 5. Reproject DEM to GUNW EPSG

In [ ]:
from osgeo import gdal
gdal.UseExceptions()

input_raster = dem_4326
output_raster = data_dir / f"DEM_{epsg}.tif"
target_crs = f"EPSG:{epsg}"

warp_options = gdal.Warp(output_raster, input_raster, dstSRS=target_crs)
warp_options = None

dem_4326.unlink()